In [1]:
!pip install -qU pinecone sentence-transformers pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 30.3 MB/s eta 0:00:00


In [2]:
import os
import uuid
from typing import List

from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from google.colab import files

In [3]:
# Pinecone config
PINECONE_API_KEY = "my api key"
PINECONE_INDEX_NAME = "assunnah-index"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# Embedding model
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"
EMBEDDING_DIM = 1024

# Chunking config
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

In [4]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION
        )
    )

index = pc.Index(PINECONE_INDEX_NAME)
print(f"Connected to index: {PINECONE_INDEX_NAME}")

Connected to index: assunnah-index


In [5]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded.


In [6]:
def read_txt(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)

def read_docx(file_path: str) -> str:
    doc = Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])

def load_document(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".txt":
        return read_txt(file_path)
    elif ext == ".pdf":
        return read_pdf(file_path)
    elif ext == ".docx":
        return read_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

In [7]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = text.replace("\t", " ").replace("\r", " ")
    text = " ".join(text.split())

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [8]:
def embed_documents(texts: List[str]) -> List[List[float]]:
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    return embeddings.tolist()

In [9]:
uploaded = files.upload()
file_paths = list(uploaded.keys())
print("Uploaded files:", file_paths)

Saving as_sunnah_foundation_extract.txt to as_sunnah_foundation_extract.txt
Uploaded files: ['as_sunnah_foundation_extract.txt']


In [10]:
def ingest_documents(file_paths: List[str]):
    all_vectors = []

    for file_path in file_paths:
        print(f"\nProcessing file: {file_path}")

        text = load_document(file_path)
        if not text.strip():
            print(f"Skipping empty file: {file_path}")
            continue

        chunks = chunk_text(text)
        print(f"Total chunks: {len(chunks)}")

        embeddings = embed_documents(chunks)

        for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
            vector_id = str(uuid.uuid4())

            metadata = {
                "source": file_path,
                "chunk_id": i,
                "text": chunk
            }

            all_vectors.append({
                "id": vector_id,
                "values": emb,
                "metadata": metadata
            })

    if not all_vectors:
        print("No vectors generated.")
        return

    batch_size = 100
    for i in range(0, len(all_vectors), batch_size):
        batch = all_vectors[i:i + batch_size]
        index.upsert(vectors=batch)

    print(f"\nUploaded {len(all_vectors)} chunks to Pinecone.")

In [11]:
ingest_documents(file_paths)


Processing file: as_sunnah_foundation_extract.txt
Total chunks: 23


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Uploaded 23 chunks to Pinecone.


In [12]:
def embed_query(query: str) -> List[float]:
    query_instruction = f"Represent this sentence for searching relevant passages: {query}"
    embedding = embedding_model.encode(
        [query_instruction],
        normalize_embeddings=True
    )[0]
    return embedding.tolist()

In [13]:
def search_pinecone(query: str, top_k: int = 5):
    query_vector = embed_query(query)
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    return results

In [14]:
query = "can you explain about dawah activities?"
results = search_pinecone(query)

for match in results["matches"]:
    print("=" * 80)
    print("Score:", match["score"])
    print("Source:", match["metadata"]["source"])
    print("Chunk ID:", match["metadata"]["chunk_id"])
    print("Text:", match["metadata"]["text"][:500])

Score: 0.715584815
Source: as_sunnah_foundation_extract.txt
Chunk ID: 11
Text: ion’s daʼwah initiatives aim to inspire Islamic values by spreading authentic knowledge. Activities include producing video lectures and Q&A sessions, distributing books and leaflets, organising workshops, running a daʼwah diploma programme, offering shariah solutions through live Q&A and an open call centre, and providing in‑person consultations and family counselling. Impact statistics for this project record, among other things, over **256 daʼwah videos** reaching more than **30 million viewe
Score: 0.673873961
Source: as_sunnah_foundation_extract.txt
Chunk ID: 1
Text: For Ummah, With Sunnah”** summarises the foundation’s focus areas: - **Education:** The foundation establishes and manages madrasas with integrated syllabuses combining religious and general education to produce distinguished Islamic scholars and contemporary daʼwah practitioners. - **Service:** It empowers the poor, provides relief and reh